# Combine Seed Plots

This notebook iterates through a specific folder structure to combine corresponding PNG images vertically. Images are expected to be in folders that end with a number specified in `seeds_to_merge`.

In [3]:
import os
import glob
from PIL import Image
from collections import defaultdict
import re

seeds_to_merge = [-1, 0, 1, 2, 3, 7]

# --- Paths ---
INPUT_BASE_DIR = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score"
OUTPUT_DIR = r"C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds"


# create new folders

In [2]:
protein_names = ["ASCT2", "CCR5", "CGRPR", "FZD7", "LAT1", "MCT1", "MurJ", "PfMATE", "PTH1R", "SERT", "STP10" , "ZnT8"]
new_folder_blueprint = "qu_mask_15_<X>_AlaRepSEEDchange-1"

for protein in protein_names:
    # Replace <X> with the specific protein name
    folder_name = new_folder_blueprint.replace("<X>", protein)
    
    # Construct the full path
    folder_path = os.path.join(INPUT_BASE_DIR, folder_name)
    
    # Create the folder (exist_ok=True prevents errors if the folder already exists)
    os.makedirs(folder_path, exist_ok=True)
    
    print(f"Created folder: {folder_path}")


Created folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\qu_mask_15_ASCT2_AlaRepSEEDchange-1
Created folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\qu_mask_15_CCR5_AlaRepSEEDchange-1
Created folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\qu_mask_15_CGRPR_AlaRepSEEDchange-1
Created folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\qu_mask_15_FZD7_AlaRepSEEDchange-1
Created folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\qu_mask_15_LAT1_AlaRepSEEDchange-1
Created folder: C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\TM_Score\qu_mask_15_MCT1_AlaRepSEEDchange-1
Created folder: C:\Users\franc\local_files(non_ODr

# Rest

In [4]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Group folders by their base name (excluding the last numbers)
folder_groups = defaultdict(dict)

for folder_name in os.listdir(INPUT_BASE_DIR):
    folder_path = os.path.join(INPUT_BASE_DIR, folder_name)
    if os.path.isdir(folder_path):
        # Extract trailing number
        match = re.search(r'(-?\d+)$', folder_name)
        if match:
            seed_idx = int(match.group(1))
            base_name = folder_name[:-len(match.group(1))]
            folder_groups[base_name][seed_idx] = folder_path
        else:
            pass # ignore folders that don't end in a number


In [6]:
# Process each group
for base_name, paths_dict in folder_groups.items():
    # Only process if we have at least the required seeds (can be more)
    if set(seeds_to_merge).issubset(paths_dict.keys()):
        images = []
        valid = True
        
        # Load images in the exact order specified by seeds_to_merge
        for i in seeds_to_merge:
            folder_path = paths_dict[i]
            # Find the single .png file in the folder
            png_files = glob.glob(os.path.join(folder_path, "*.png"))
            if len(png_files) != 1:
                print(f"Warning: Expected exactly 1 PNG file in {folder_path}, found {len(png_files)}. Skipping group.")
                valid = False
                break
            
            img_path = png_files[0]
            try:
                img = Image.open(img_path)
                images.append(img)
            except Exception as e:
                print(f"Error opening image {img_path}: {e}")
                valid = False
                break
                
        if not valid:
            continue
            
        # Combine images vertically
        widths, heights = zip(*(i.size for i in images))
        
        max_width = max(widths)
        total_height = sum(heights)
        
        new_im = Image.new('RGB', (max_width, total_height), color='white')
        
        y_offset = 0
        for im in images:
            new_im.paste(im, (0, y_offset))
            y_offset += im.size[1]
            
        # Determine output filename
        # Naming of the output files: the 4-times repeating folder names up to (excluding) the 4th occurence of an underscore "_"
        # e.g., qu_mask_15_ASCT2_AlaRepSEEDchange -> qu_mask_15_ASCT2
        parts = base_name.split("_")
        if len(parts) > 4:
            out_name = "_".join(parts[:4])
        else:
            out_name = base_name
            
        out_path = os.path.join(OUTPUT_DIR, f"{out_name}.png")
        new_im.save(out_path)
        print(f"Saved combined image to {out_path}")


Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\qu_mask_15_ASCT2.png
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\qu_mask_15_CCR5.png
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\qu_mask_15_CGRPR.png
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\qu_mask_15_FZD7.png
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\qu_mask_15_LAT1.png
Saved combined image to C:\Users\franc\local_files(non_ODrive)\protPred_colab_runs\AlaRep\plotting_input_and_output\plots\combined_plots_for_seeds\qu_mask_15_MCT1.